# Streaming with intervention

The agent runs as a generator that yields `{node_complete, ...}` events. When the policy fires an `interrupt`, the generator yields one more event and stops — the UI shows a prompt, the user supplies the resume value, and a second call streams the remainder.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / '04-human-in-the-loop'))
os.environ.setdefault('LLM_CACHE_ONLY', '1')

from hitl import build_research_graph, Checkpointer, Command, get_question, stream_events

graph = build_research_graph(); cp = Checkpointer()
stream = stream_events(graph, {'question': get_question('q01'), 'question_id': 'q01'},
                       thread_id='live', checkpointer=cp)
for event in stream:
    print(event['event'], event.get('node'))
    if event['event'] == 'interrupt':
        print('paused with:', event['payload'])
        break

In [ ]:
# In a real UI: this is the POST that closes the loop.
state = graph.resume(thread_id='live',
                     command=Command(resume={'approved': True, 'reviewer': 'live-user'}),
                     checkpointer=cp)
print('final answer:', state['final_answer'])
print('checkpoints :', len(cp.history('live')))

## Why a generator (not callbacks)

A generator inverts control: **the agent yields, the caller drives the loop.** The caller can buffer events, retry, abort, or wait for input — all without the agent caring about the transport. Wrap the generator in an SSE response (the next deployment-patterns folder does this with FastAPI) and you have streaming HITL for free.